In [1]:
# In họ tên và MSSV
print("Họ và tên: Trần Minh Tấn")
print("MSSV: 6451071071")

import pandas as pd
import numpy as np
import warnings
import io
import requests

# Tắt toàn bộ các cảnh báo rác
warnings.filterwarnings('ignore')

# 1. Đọc dữ liệu Bank Marketing từ Google Drive qua ID chuẩn xác
file_id = '1X61cZvrgBBYgBM1KSnFc-7wRO5NZcOns'
download_url = f'https://drive.google.com/uc?export=download&id={file_id}'
response = requests.get(download_url)

try:
    df = pd.read_csv(io.BytesIO(response.content))
except Exception:
    # Trường hợp file có định dạng phân tách khác (ví dụ dấu chấm phẩy ;)
    df = pd.read_csv(io.BytesIO(response.content), sep=';')

print("--- 1. THÔNG TIN TỔNG QUAN VỀ DỮ LIỆU ---")
print(f"Kích thước tập dữ liệu (Số dòng, Số cột): {df.shape}")
display(df.head())

print("\n--- 2. KIỂM TRA KIỂU DỮ LIỆU VÀ GIÁ TRỊ THIẾU (MISSING VALUES) ---")
display(df.info())
print("\nSố lượng giá trị thiếu trên mỗi cột:")
display(df.isnull().sum())

print("\n--- 3. TIỀN XỬ LÝ VÀ CHUẨN HÓA DỮ LIỆU ---")
# Thay thế các giá trị 'unknown' thành NaN để dễ xử lý
df.replace('unknown', np.nan, inplace=True)

# Điền giá trị khuyết thiếu cho các biến phân loại bằng giá trị xuất hiện nhiều nhất (Mode)
categorical_cols = ['Job', 'Marial', 'Education', 'Default', 'Loan']
for col in categorical_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

# Điền giá trị khuyết thiếu cho các biến số bằng số Trung vị (Median)
numerical_cols = ['Duration', 'Cons.price.idx']
for col in numerical_cols:
    if col in df.columns and df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)

print("Đã xử lý xong giá trị thiếu!")
display(df.isnull().sum())

print("\n--- 4. MÃ HÓA DỮ LIỆU (ENCODING) CHO CÁC BIẾN PHÂN LOẠI ---")
# Chuyển đổi các cột dạng chữ (categorical) sang dạng số để đưa vào mô hình học máy
df_encoded = pd.get_dummies(df, columns=[c for c in categorical_cols if c in df.columns], drop_first=True)

# Chuẩn hóa nhãn mục tiêu 'Subscribe' (yes/no thành 1/0 nếu có)
if 'Subscribe' in df_encoded.columns:
    df_encoded['Subscribe'] = df_encoded['Subscribe'].map({'yes': 1, 'no': 0})

print("Kích thước dữ liệu sau khi mã hóa One-Hot Encoding:", df_encoded.shape)
display(df_encoded.head())

Họ và tên: Trần Minh Tấn
MSSV: 6451071071
--- 1. THÔNG TIN TỔNG QUAN VỀ DỮ LIỆU ---
Kích thước tập dữ liệu (Số dòng, Số cột): (4521, 1)


,"age;""job"";""marital"";""education"";""default"";""balance"";""housing"";""loan"";""contact"";""day"";""month"";""duration"";""campaign"";""pdays"";""previous"";""poutcome"";""y"""
0,"30;""unemployed"";""married"";""primary"";""no"";1787;..."
1,"33;""services"";""married"";""secondary"";""no"";4789;..."
2,"35;""management"";""single"";""tertiary"";""no"";1350;..."
3,"30;""management"";""married"";""tertiary"";""no"";1476..."
4,"59;""blue-collar"";""married"";""secondary"";""no"";0;..."



--- 2. KIỂM TRA KIỂU DỮ LIỆU VÀ GIÁ TRỊ THIẾU (MISSING VALUES) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 1 columns):
 #   Column                                                                                                                                                Non-Null Count  Dtype 
---  ------                                                                                                                                                --------------  ----- 
 0   age;"job";"marital";"education";"default";"balance";"housing";"loan";"contact";"day";"month";"duration";"campaign";"pdays";"previous";"poutcome";"y"  4521 non-null   object
dtypes: object(1)
memory usage: 35.4+ KB


None


Số lượng giá trị thiếu trên mỗi cột:


,0
"age;""job"";""marital"";""education"";""default"";""balance"";""housing"";""loan"";""contact"";""day"";""month"";""duration"";""campaign"";""pdays"";""previous"";""poutcome"";""y""",0



--- 3. TIỀN XỬ LÝ VÀ CHUẨN HÓA DỮ LIỆU ---
Đã xử lý xong giá trị thiếu!


,0
"age;""job"";""marital"";""education"";""default"";""balance"";""housing"";""loan"";""contact"";""day"";""month"";""duration"";""campaign"";""pdays"";""previous"";""poutcome"";""y""",0



--- 4. MÃ HÓA DỮ LIỆU (ENCODING) CHO CÁC BIẾN PHÂN LOẠI ---
Kích thước dữ liệu sau khi mã hóa One-Hot Encoding: (4521, 1)


,"age;""job"";""marital"";""education"";""default"";""balance"";""housing"";""loan"";""contact"";""day"";""month"";""duration"";""campaign"";""pdays"";""previous"";""poutcome"";""y"""
0,"30;""unemployed"";""married"";""primary"";""no"";1787;..."
1,"33;""services"";""married"";""secondary"";""no"";4789;..."
2,"35;""management"";""single"";""tertiary"";""no"";1350;..."
3,"30;""management"";""married"";""tertiary"";""no"";1476..."
4,"59;""blue-collar"";""married"";""secondary"";""no"";0;..."
